In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import pickle
import numpy as np
import pandas as pd
from src.common.utils.config import load_config
from src.modeling.feature_selector import get_feature_columns
from src.modeling.splitter import time_split
from src.modeling.metrics import mape, wmape

In [2]:
# Locate the model — it lives in MLflow artifacts (Cell 8 of notebook 4 wrote it there)
MODEL_PKL = Path("../pipelines/6_modeling/mlruns/222454018282762985/6e6f9101d861404396c3195738fa2dd8/artifacts/model_3_high-value_active_continuous.pkl")
CELL_DATA  = Path("../data/Intermediate/cell_3_high-value_active_continuous.parquet")

print("Model exists :", MODEL_PKL.exists(), "|", MODEL_PKL.resolve())
print("Cell data    :", CELL_DATA.exists(), "|", CELL_DATA.resolve())

Model exists : True | /Users/mohamedinas/Desktop/SE_projects/8_stax_interview/8_interview_prep/pipelines/6_modeling/mlruns/222454018282762985/6e6f9101d861404396c3195738fa2dd8/artifacts/model_3_high-value_active_continuous.pkl
Cell data    : True | /Users/mohamedinas/Desktop/SE_projects/8_stax_interview/8_interview_prep/data/Intermediate/cell_3_high-value_active_continuous.parquet


In [3]:
with open(MODEL_PKL, "rb") as f:
    reg = pickle.load(f)

print(type(reg))
print(reg)

<class 'xgboost.sklearn.XGBRegressor'>
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device='cpu', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.01, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, nthread=1, ...)


In [4]:
cfg     = load_config()
mod_cfg = cfg["modeling"]

df = pd.read_parquet(CELL_DATA)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
df.head(3)

Rows: 2,829,292 | Columns: 57


,outlet_id,item_code,year_month,net_sales,return_quantity,return_sales,n_invoices,n_returns,territory_id,region,...,target_qty_log1p,target_is_nonzero,target_log1p_net_sales,demand_class,demand_class_encoded,sku_activity_rate,is_structural_collapse,is_hyper_growth,sku_log_total_qty,seasonal_index
0,10000,gf1000203,2015-07,564.0,0,0.0,1,0,29,region_6,...,0.0,0.0,0.0,Continuous,0,1.0,0,0,14.53719,0.944962
1,10000,gf1000203,2015-08,0.0,0,0.0,0,0,29,region_6,...,0.0,0.0,0.0,Continuous,0,1.0,0,0,14.53719,0.937863
2,10000,gf1000203,2015-09,0.0,0,0.0,0,0,29,region_6,...,0.0,0.0,0.0,Continuous,0,1.0,0,0,14.53719,0.994775


In [5]:
df_train, df_eval = time_split(df, mod_cfg["train_end"], mod_cfg["eval_start"])
print(f"Train: {len(df_train):,} | Eval: {len(df_eval):,}")

target_col = mod_cfg["target_regressor"]
feature_cols, target_col = get_feature_columns(df_train, target_col, mod_cfg["drop_columns"])
print(f"Features: {len(feature_cols)}")

Train: 2,278,174 | Eval: 551,118
Features: 41


In [6]:
X_eval  = df_eval[feature_cols].values
y_true  = df_eval[mod_cfg["target_regressor"]].values
y_pred  = reg.predict(X_eval)

mape_val, zero_frac = mape(y_true, y_pred)
wmape_val = wmape(y_true, y_pred)

print(f"MAPE  : {mape_val:.2f}%  (zero actuals excluded: {zero_frac:.1%})")
print(f"WMAPE : {wmape_val:.2f}%")

MAPE  : 68.56%  (zero actuals excluded: 85.1%)
WMAPE : 157.28%


In [7]:
# Sample predictions vs actuals
results = df_eval[["outlet_id", "item_code", "year_month", mod_cfg["target_regressor"]]].copy()
results["predicted"] = y_pred
results["error"]     = results["predicted"] - results[mod_cfg["target_regressor"]]
results.head(10)

,outlet_id,item_code,year_month,target_qty_raw,predicted,error
12,10000,gf1000203,2019-05,0.0,0.733738,0.733738
13,10000,gf1000203,2019-06,0.0,0.663146,0.663146
14,10000,gf1000203,2019-07,0.0,0.733738,0.733738
15,10000,gf1000203,2019-08,0.0,0.827435,0.827435
16,10000,gf1000203,2019-09,0.0,0.827435,0.827435
17,10000,gf1000203,2019-10,0.0,0.652836,0.652836
66,10000,tt0060101,2019-10,0.0,2.589571,2.589571
73,10000,wf2031901,2019-09,0.0,6.050875,6.050875
74,10000,wf2031901,2019-10,0.0,5.390366,5.390366
106,10001,pf0084103,2019-02,0.0,0.204189,0.204189
